# 08 · Score the sealed test once, then decide H1a–H1c and H2

Sandesh Lamichhane, Assistant Professor, Yenepoya (Deemed to be University).

**Run notebook 07 first.** This is the one-time test scoring of the sealed-test design (ADR-0007; procedure in
preregistration §6b, fixed before any test score existed). It:

1. verifies every processed dataset and S7 file against its SHA-256 (step 2);
2. checks both freezes (Levels 1–2 and Level 3 under the 1.6 rules) before anything is read (step 3);
3. scores **every frozen model**, the proposed detector and all baselines, five seeds each, at its frozen
   validation threshold, on the CICIDS2017 test partition, UNSW-NB15, CSE-CIC-IDS2018 and LANL days 14–58
   (steps 4–7). Nothing is fitted or re-tuned;
4. runs the preregistered analysis once and writes the decisions (step 8).

**There is no "expected result" for the decisions.** Nobody has seen a test number yet. Whatever step 8 prints is
the result, and it is reported as it comes, including "not supported". If a step stops with an error, do not
change anything: copy the error into the project thread.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, and switch **High-RAM** on. The MLP and CNN-LSTM
baselines need the GPU; CSE-CIC-IDS2018 (45 million rows) and the 58-day LANL table need High-RAM. Step 6 also
writes about 20 GB of temporary files to the Colab disk (not Drive), which is deleted when the runtime ends.

**Interruptions are safe.** Every scored unit (dataset, feature space, model, seed) is saved to Drive the moment it
finishes, and is never scored twice. After a disconnect, run step 1 again, then the step that was interrupted: it
continues where it stopped. Step 8 refuses to run until every unit exists, and runs only once: running it again
recomputes everything and stops if any number differs.

| Step | What it does | Approximate time |
|---|---|---|
| 1 | connect Drive and GitHub, update the code, run the tests (expected `… passed`, no failures) | 4 min |
| 2 | verify all processed data and S7 files (expected `all outputs verified`) | 20–40 min |
| 3 | check both freezes (expected two `OK` lines) and list what will be scored | 5–20 min |
| 4 | score the CICIDS2017 test partition (5 models x 5 seeds x 2 feature spaces) | 10–30 min |
| 5 | score UNSW-NB15 (5 models x 5 seeds) | 15–40 min |
| 6 | score CSE-CIC-IDS2018 (5 models x 5 seeds, 45 M rows) | 2–5 h |
| 7 | score LANL days 14–58 (4 models x 5 seeds) | 15–40 min |
| 8 | the confirmatory analysis: H1a–H1c, H2, S7, Holm → decisions | 5–15 min |

**Where the results go.** In the Drive copy of the repository (`MyDrive/PhD-AI-Cyber/code/phd-ai-threat-detection/`)
and on GitHub. There is nothing to move by hand.

* `results/confirmatory/decisions.md`: **the result**, as tables for the paper and the thesis (decisions with
  95 % CIs, Holm-adjusted p-values, every baseline, S7, the Level 3 baselines)
* `results/confirmatory/decisions.json`, `level1_2.json`, `level3.json`: the same numbers in full
* `results/confirmatory/level12/` and `level3/`: one record per scored unit (alerts per block and label, with the
  SHA-256 of the model, the data and the freeze it used)
* `results/confirmatory/logs/<time>_….log`: run logs

Per-row scores of the proposed detector go to `MyDrive/PhD-AI-Cyber/scores/` (Drive only; their SHA-256 values
are in the unit records on GitHub).

**When it finishes:** use *File → Download → Download .ipynb* and send the finished notebook to the project thread.

## Step 1 · Connect and update (expected end: `in sync with GitHub at …` and `… passed` with no failures)

Always run this cell first, also after a disconnect. It loads the newest code.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import datetime as _dt, json, os, pathlib, shutil, subprocess, time

DRIVE_ROOT = '/content/drive/MyDrive/PhD-AI-Cyber'
REPO_DIR   = f'{DRIVE_ROOT}/code/phd-ai-threat-detection'
os.environ['PHD_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['PHD_SCRATCH'] = '/content/scratch'
TOKEN  = userdata.get('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/phd-ai-threat-detection.git'

# Git keeps its database on the Colab disk, never on Drive: Drive's file system can crash git while it unpacks
# downloads ("unpack-objects died of signal 7"). The files themselves stay in the Drive folder, so every result
# is in Drive the moment it is written, and the sync below copies it to GitHub.
GIT_DIR = '/content/phd-repo.git'
STATE = pathlib.Path(DRIVE_ROOT, 'code', '.phd_sync_state.json')    # commit the Drive folder last matched
os.environ['GIT_DIR'], os.environ['GIT_WORK_TREE'] = GIT_DIR, REPO_DIR
RECORD_DIRS = ('results/', 'docs/data_cards/', 'config/feature_mapping/')
_trust_drive_records = True

def _mask(text):
    return text.replace(TOKEN or '\0', '***').strip()

def git(*args, check=False):
    r = subprocess.run(['git', *args], capture_output=True, text=True, cwd=REPO_DIR)
    out = _mask(r.stdout + r.stderr)
    if check and r.returncode != 0:
        print(out)
        raise SystemExit(f'git {args[0]} failed (see message above)')
    return r.returncode, out

def _stamp():
    return _dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')

def _last_synced_commit():
    """The commit the Drive folder was last synced to: from the state file, else from the old .git on Drive."""
    try:
        if STATE.exists():
            return json.loads(STATE.read_text()).get('commit')
        g = pathlib.Path(REPO_DIR, '.git')
        head = (g / 'HEAD').read_text().strip()
        if not head.startswith('ref: '):
            return head
        ref = head[5:]
        if (g / ref).exists():
            return (g / ref).read_text().strip()
        for line in (g / 'packed-refs').read_text().splitlines():
            if line.endswith(' ' + ref):
                return line.split()[0]
    except (OSError, ValueError):
        pass
    return None

def setup_git():
    """Once per Colab session: download the git database to the Colab disk and line it up with the Drive folder."""
    global _trust_drive_records
    if pathlib.Path(GIT_DIR, 'HEAD').exists():
        return
    pathlib.Path(REPO_DIR).mkdir(parents=True, exist_ok=True)
    env = {k: v for k, v in os.environ.items() if k not in ('GIT_DIR', 'GIT_WORK_TREE')}
    r = subprocess.run(['git', 'clone', '--bare', '--quiet', REMOTE, GIT_DIR], capture_output=True, text=True, env=env)
    if r.returncode != 0:
        print(_mask(r.stdout + r.stderr))
        raise SystemExit('git clone failed (see message above)')
    git('config', 'core.bare', 'false', check=True)
    git('config', 'core.filemode', 'false')
    old = _last_synced_commit()
    known = bool(old) and git('cat-file', '-e', f'{old}^{{commit}}')[0] == 0
    git('reset', '-q', old if known else 'HEAD', check=True)      # sets the index only; Drive files untouched
    _trust_drive_records = known
    if not known:
        print('note: could not tell which commit the Drive folder last matched; GitHub wins for every tracked '
              'file, and Drive versions that differ are backed up first')
    dot = pathlib.Path(REPO_DIR, '.git')
    if dot.exists():
        dest = pathlib.Path(DRIVE_ROOT, 'backups', f'old_drive_git_{_stamp()}')
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(dot), str(dest))
        print(f'old git folder on Drive moved to {dest} (no longer used)')
    print('git database on the Colab disk; Drive folder last matched ' + (old[:7] if known else 'an unknown commit'))

def _push_hint(out):
    o = out.lower()
    if any(k in o for k in ('authentication failed', 'invalid username', 'invalid credentials', '403', 'permission')):
        return ('GitHub did not accept the token. If it has expired, create a new fine-grained token (repository '
                'phd-ai-threat-detection, Contents: Read and write) and save it as the Colab secret GITHUB_TOKEN.')
    if 'rejected' in o or 'fetch first' in o or 'non-fast-forward' in o:
        return 'GitHub had a newer commit; fetching it and trying again.'
    return 'GitHub or the network did not respond; trying again in a moment.'

def sync(message=None, strict=True, also=()):
    """Make the Drive copy of the repo identical to GitHub, keeping every result produced here.

    Never merges or rebases, so open notebooks or other local edits cannot block it:
    1. remember result files changed here (results/, docs/data_cards/, config/feature_mapping/)
    2. back up every other local change to MyDrive/PhD-AI-Cyber/backups/sync_<time>/
    3. reset the copy to GitHub's main, put the result files back, commit and push them
    Results are on Drive before this runs. If GitHub cannot be reached, strict=False only warns: the next
    sync (the next step, or step 1 in a new session) saves the same files.
    `also` names other files changed here on purpose (e.g. docs/protocol/prior_exposure.md), saved like results.
    """
    global _trust_drive_records
    setup_git()
    git('config', 'user.name', 'Sandesh Lamichhane')
    git('config', 'user.email', 'san30lamichhane@gmail.com')
    tries, out = 5, ''
    for attempt in range(1, tries + 1):
        code, out = git('fetch', '--quiet', REMOTE, 'main')
        if code == 0:
            base = git('merge-base', 'HEAD', 'FETCH_HEAD')[1] or 'FETCH_HEAD'
            changed = set(git('diff', '--name-only', base)[1].splitlines())
            untracked = set(git('ls-files', '--others', '--exclude-standard')[1].splitlines())
            # untracked non-record files survive the reset unless GitHub has a file at the same path
            untracked = {f for f in untracked if f.startswith(RECORD_DIRS) or git('cat-file', '-e', f'FETCH_HEAD:{f}')[0] == 0}
            changed |= untracked
            is_file = lambda f: pathlib.Path(REPO_DIR, f).is_file()
            keep = sorted(f for f in changed if is_file(f) and (f in also or f.startswith(RECORD_DIRS)
                          and (_trust_drive_records or f in untracked)))
            other = sorted(f for f in changed if f not in keep and is_file(f))
            stamp = _stamp()
            hold = pathlib.Path('/content/sync_hold') / stamp
            for f in keep:
                (hold / f).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(pathlib.Path(REPO_DIR, f), hold / f)
            if other:
                bak = pathlib.Path(DRIVE_ROOT, 'backups', f'sync_{stamp}')
                for f in other:
                    (bak / f).parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(pathlib.Path(REPO_DIR, f), bak / f)
                print(f'{len(other)} other local change(s) backed up to {bak} (e.g. notebook outputs)')
            git('reset', '--hard', '--quiet', 'FETCH_HEAD', check=True)
            for f in keep:
                dest = pathlib.Path(REPO_DIR, f)
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(hold / f, dest)
            if keep:
                git('add', '--', *keep, check=True)
                if git('diff', '--cached', '--quiet')[0] != 0:
                    git('commit', '-q', '-m', message or 'Save results produced in Colab', check=True)
            code, out = git('push', '--quiet', REMOTE, 'HEAD:main')
            if code == 0:
                head = git('log', '--oneline', '-1')[1]
                STATE.write_text(json.dumps({'commit': git('rev-parse', 'HEAD')[1], 'utc': stamp}))
                _trust_drive_records = True
                print(f'in sync with GitHub at {head}' + (f'; {len(keep)} result file(s) saved' if keep else ''))
                return True
        print(f'GitHub refused the save (try {attempt} of {tries}): {out or "no message"}')
        print('  ' + _push_hint(out))
        if attempt < tries:
            time.sleep(min(60, 4 * 2 ** (attempt - 1)))
    msg = ('Could not save to GitHub. Nothing is lost: every result is in the Drive folder, and the next save '
           '(the next step, or step 1 after reopening the notebook) sends it to GitHub. Copy the messages above '
           'into the project thread if this keeps happening.')
    if strict:
        raise SystemExit(msg)
    print('WARNING: ' + msg)
    return False


sync('Save results from an earlier Colab run')      # update the code; results made here are kept
assert pathlib.Path(REPO_DIR, 'src/models/score.py').exists(), 'old code: run this cell again'
%cd $REPO_DIR
!pip -q install -r requirements.txt
!python -m pytest -q tests
import sys
sys.path.insert(0, REPO_DIR)
from IPython.display import Markdown, display

def push(message, also=()):
    """Save results from the Drive copy of the repo to GitHub, so both hold the same records.
    If GitHub cannot be reached it only warns: the results are in Drive and the next save sends them."""
    if sync(message, strict=False, also=also):
        print('saved to GitHub:', message)

def score(*args, message):
    """Run scripts/score.py (its output shows below); push the records only if it succeeded."""
    ip = get_ipython()
    ip.system('python scripts/score.py ' + ' '.join(args))
    ok = ip.user_ns.get('_exit_code', 1) == 0
    push(message if ok else message + ' (FAILED, log only)')
    if not ok:
        raise SystemExit('This step failed. Copy the error above into the project thread.')

## Step 2 · Verify the data (expected: `all outputs verified`)

Checks that every processed dataset was built under the current rules and that every processed, split and S7
file matches the SHA-256 in its report (preregistration §10a). Writes `results/data_audit/verification.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import psutil
ram = psutil.virtual_memory().total / 2**30
print(f'RAM {ram:.0f} GB')
if ram < 40:
    print('WARNING: High-RAM seems to be off. Turn it on (Runtime -> Change runtime type) before step 6.')
!python scripts/preprocess.py --verify
if get_ipython().user_ns.get('_exit_code', 1) != 0:
    push('Verification before test scoring (FAILED, log only)')
    raise SystemExit('Verification failed. Copy the output above into the project thread and stop here.')
push('Verify all outputs before the one-time test scoring')

## Step 3 · Check both freezes (expected: two `OK` lines)

Expected: `OK Levels 1-2: random_forest …` and `OK Level 3 (rules 1.6): …` with the model notebook 07 froze. The
scoring code runs the same checks again before it reads any test row.

In [ ]:
from src.models import seal
for track, label in (('l12', 'Levels 1-2'), ('lanl', 'Level 3')):
    try:
        f = seal.require_frozen(track=track)
    except seal.Sealed as e:
        raise SystemExit(f'{label} is not frozen ({e}). Run notebook 07 first.')
    rules = f" (rules {f['rules_version']})" if 'rules_version' in f else ''
    print(f"OK {label}{rules}: {f['proposed']}, frozen {f['frozen_utc']}; {len(f['model_files_sha256'])} model "
          f"files and {len(f['development_records_sha256'])} development records match their SHA-256")
!python scripts/score.py --status

## Step 4 · Score the CICIDS2017 test partition (10–30 min)

Expected lines like `[cicids2017_test/cic/random_forest] seed 11: … of 221,066 rows alerted`, 50 in all, then `in sync with GitHub at …`.

In [ ]:
score('--level12', '--eval', 'cicids2017_test', message='Score the frozen models on the CICIDS2017 test partition')

## Step 5 · Score UNSW-NB15 (15–40 min)

Expected: 25 lines like `[unsw_nb15/unsw_canonical/random_forest] seed 11: … of 2,027,278 rows alerted`.

In [ ]:
score('--level12', '--eval', 'unsw_nb15', message='Score the frozen models on UNSW-NB15')

## Step 6 · Score CSE-CIC-IDS2018 (2–5 h)

First it copies the 45 million rows to the Colab disk (`cached part …` lines, 15–30 min), then 25 lines like `[csecicids2018/cic/random_forest] seed 11: … of 45,147,292 rows alerted`. If the runtime disconnects, run step 1 and this step again: finished units are kept.

In [ ]:
score('--level12', '--eval', 'csecicids2018', message='Score the frozen models on CSE-CIC-IDS2018')

## Step 7 · Score LANL days 14–58 (15–40 min)

Expected: 20 lines like `[lanl/logreg] seed 11: … of 44 test hosts alerted early, false alarms …`.

In [ ]:
score('--lanl', message='Score the frozen Level 3 models on LANL days 14-58')

## Step 8 · The confirmatory analysis (once)

Block bootstrap (B = 10 000) for H1a–H1c, bootstrap over red-team accounts for H2, Holm over the confirmatory family
at 0.025 per direction (the three H4 claims enter as p = 1 until Level 4 exists), S7, the Level 3 random and
reactive baselines, and the secondary model comparisons. It prints the decisions table at the end. This result is
final: it is reported as it is.

In [ ]:
score('--analyse', message='Confirmatory analysis of Levels 1-3: decisions for H1a-H1c and H2 (preregistration §6b)')
sync('Final save of the confirmatory results', strict=True)
display(Markdown(pathlib.Path(REPO_DIR, 'results/confirmatory/decisions.md').read_text()))
print('On GitHub: https://github.com/sandesh20lamichhane/phd-ai-threat-detection/tree/main/results/confirmatory')
print('Done. Download this notebook (File -> Download -> Download .ipynb) and send it to the project thread.')